In [0]:
# Bronze (External Volume)
bronze_aw2_path = "/Volumes/main/default/fabmigration1_bronze/adventureworks2/"

df_aw2_bronze = spark.read.parquet(bronze_aw2_path)

In [0]:
bronze_base = "/Volumes/main/default/fabmigration1_bronze/adventureworks2/"

def list_parquets(path):
    out = []
    for i in dbutils.fs.ls(path):
        if i.path.endswith("/"):
            out.extend(list_parquets(i.path))
        elif i.path.lower().endswith(".parquet"):
            out.append(i.path)
    return out

parquets = list_parquets(bronze_base)

hr_files = [p for p in parquets if "/HumanResources." in p or p.split("/")[-1].startswith("HumanResources.")]
print(f"Found {len(hr_files)} parquet files under HumanResources.*")
for p in hr_files[:50]:
    print(" -", p)

In [0]:
from pyspark.sql import functions as F

def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls (solo si hay columnas de ese tipo)
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {
        col: 0
        for col, dtype in df.dtypes
        if dtype in ["int", "bigint", "double", "float"]
    }

    if fill_dict:                 # <- evita fillna({}) que truena
        df = df.fillna(fill_dict)

    if fill_dict_numeric:         # <- evita fillna({}) que truena
        df = df.fillna(fill_dict_numeric)

    return df

In [0]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T

def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",     # "date" or "timestamp"
    ingest_date: Optional[str] = None  # e.g., "2026-02-03" (casts to date); if None -> current_date()
) -> DataFrame:
    """
    1) Standardizes the provided or auto-detected date/timestamp columns:
       - Parses common string formats to timestamp
       - Casts to `date` (default) or keeps as `timestamp` via `cast_dates_as`
    2) Adds `ingest_date` column (date) for downstream partitioning.

    Parameters
    ----------
    df : input DataFrame
    date_cols : list of column names to treat as dates/timestamps. If None, auto-detects by name/ctype.
    input_formats : list of strptime-style formats to try in order
    cast_dates_as : "date" or "timestamp"
    ingest_date : string literal yyyy-MM-dd; if None uses current_date()
    """

    # --- 0) Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # --- 1) Which columns are date-like?
    if date_cols is None:
        # Heuristic: names that look like dates & are string/date/timestamp
        keywords = ("date", "dt", "_at", "_on", "timestamp", "ts", "fecha", "created", "updated")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp") and any(k in c.lower() for k in keywords)
        ]

    # --- 2) Parse formats for string dates
    if input_formats is None:
        input_formats = [
            # Date only
            "yyyy-MM-dd", "yyyy/MM/dd", "dd-MM-yyyy", "dd/MM/yyyy", "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",
            # Date + time
            "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss",
            # ISO-ish / with millis / TZ
            "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssXXX", "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)
    for c in date_cols:
        t = dtype_map.get(c, "")

        # If it's already timestamp/date, just unify type later
        if t == "string":
            # Try multiple formats with coalesce
            parsed = None
            for fmt in input_formats:
                candidate = F.to_timestamp(F.col(c), fmt)
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # Epoch seconds / millis (string or numeric) fallbacks
            parsed = F.coalesce(
                parsed,
                F.to_timestamp(F.col(c).cast("double")),                    # epoch seconds
                F.to_timestamp((F.col(c).cast("double")/1000.0))            # epoch millis
            )

            # If nothing matched, keep original to avoid data loss
            df = df.withColumn(c, F.when(parsed.isNotNull(), parsed).otherwise(F.col(c)))

        # Finally, cast to the requested final type
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.to_timestamp(F.col(c)))

    # --- 3) Add ingest_date for partitioning
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df

In [0]:
for file_path in hr_files:
    print(f"\nProcessing: {file_path}")

    # Normaliza el nombre (sirve si file_path es archivo o carpeta)
    base = file_path.rstrip("/").split("/")[-1]          # ej: HumanResources.Employee OR HumanResources.Employee.parquet
    base = base.replace(".parquet", "")                  # ej: HumanResources.Employee
    silver_table_name = base.replace(".", "_").lower()

    # Read parquet (si es carpeta con parquets adentro, esto funciona; si tienes subniveles, agrega recursiveFileLookup)
    df = spark.read.parquet(file_path)
    df.printSchema()

    # Clean
    df_clean = clean_df(df)

    # dates
    df_table= standardizedate_df(df_clean)

    full_table_name = f"main.silver.{silver_table_name}"
    print(f"Saving as Silver table: {full_table_name}")

    # Save as Delta table (Silver)
    df_table.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_table_name)
    